# Nature Data Cube — Guided Exploration
### Bird nest boxes meet greenness (NDVI) on the Veluwe

<div style="background:#f0f7f4;border-left:5px solid #1b7f5a;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#1b7f5a">About this session</b><br>
This is <b>not</b> a coding exercise. It is a guided tour of the Nature Data Cube (NDC) and the data it produces, using a real nest-box dataset as motivation. As you go, we ask for your feedback so we can see whether you find the NDC <b>useful, understandable, and intuitive</b>.<br><br><b>How to use:</b> run each grey code cell in order. Charts and maps appear right below the cell that makes them.
</div>

## 0 · Setup
Run this once. It loads the tools, points R at the data files, and sets a consistent look for every figure.

In [ ]:
# First time only: install the packages (uncomment and run once)
# install.packages(c("ggplot2","lubridate","sf","terra","leaflet","scales","geojsonio"))

suppressPackageStartupMessages({
  library(ggplot2)    # plotting
  library(lubridate)  # dates
  library(sf)         # vector / polygon
  library(terra)      # NDVI raster
  library(leaflet)    # interactive maps (also provides the %>% pipe)
  library(scales)
  library(geojsonio)  # read/write GeoJSON
})

# Where the downloaded NDC data lives (path to the data in MinIO)
data_dir <- "~/Cloud Storage/naa-vre-public/vl-nature-data-cube"
f_nest    <- file.path(data_dir, "nest_data.csv")
f_summary <- file.path(data_dir, "download_summary.csv")
f_ndvi    <- file.path(data_dir, "ndvi_statistics_nestboxes_2.csv")
f_polygon <- file.path(data_dir, "nestboxes_2.gpkg")
f_raster  <- file.path(data_dir, "ndvi_geodata_nestboxes_2.tif")

# Colour-blind-safe palette for the three species
species_pal <- c("blue tit" = "#0072B2", "great tit" = "#E69F00",
                 "pied flycatcher" = "#009E73")

# A clean, shared look for every chart
theme_ndc <- function() {
  theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"),
          plot.subtitle = element_text(colour = "grey35"),
          panel.grid.minor = element_blank(),
          legend.position = "bottom")
}

# Default size for static plots in the notebook
options(repr.plot.width = 9, repr.plot.height = 5.5)

## 1 · The ecological question
Songbirds time their breeding to the spring flush of vegetation, because that green-up drives the caterpillar peak their chicks depend on. So we ask:

<div style="background:#f4f0fa;border-left:5px solid #6a4fa3;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#6a4fa3">Our question</b><br>
Do birds breed <b>earlier</b> in years when the landscape greens up <b>more</b>, compared to less-green years?
</div>
We can't answer that from the birds alone — we need a measure of how green each spring was. That is exactly what the Nature Data Cube provides. Keep this question in mind; everything below builds toward answering it in Section 8.

## 2 · Background in one breath

| Term | Meaning |
|---|---|
| **Nest box** | Artificial cavity; here at fixed locations on the Veluwe. |
| **Lay date** | Day the first egg is laid (also given as day-of-year, `ld_doy`). |
| **Clutch** | Number of eggs in one nesting attempt. |
| **NDVI** | Satellite "greenness", ~0 (bare) to ~1 (dense vegetation). |

*The idea we are testing:* greener springs → more insect food → birds may breed earlier and/or lay larger clutches in those years.

## 3 · Explore the nest-box data — *meet the birds*
First we get to know the birds on their own: how much data we have, what the clutches look like, and — crucially — **when** they lay. This is the *response* side of our question; greenness comes later.

In [ ]:
nest <- read.csv(f_nest, stringsAsFactors = FALSE)
nest$lay_date <- as.Date(nest$lay_date)
nest$species  <- factor(nest$species, levels = names(species_pal))

str(nest)   # 2,065 records | 2019-2025 | 439 boxes | 3 species

### 3a · Records per year
Is monitoring effort steady? If so, later year-to-year differences reflect the **birds**, not how hard we looked.

In [ ]:
year_counts <- as.data.frame(table(year = nest$year, species = nest$species),
                             responseName = "n")
year_counts$year    <- as.integer(as.character(year_counts$year))
year_counts$species <- factor(year_counts$species, levels = names(species_pal))

p_year <- ggplot(year_counts, aes(year, n, fill = species)) +
  geom_col() +
  scale_fill_manual(values = species_pal, name = NULL) +
  scale_x_continuous(breaks = 2019:2025) +
  labs(title = "Nest records per year",
       subtitle = "Consistent monitoring effort across 2019-2025",
       x = NULL, y = "Number of nests") +
  theme_ndc()
p_year

*Effort is steady (~270–330 nests/year), so the year-to-year signals we chase later are real.*

### 3b · Clutch-size distribution
How many eggs do these species typically lay?

In [ ]:
p_clutch <- ggplot(nest, aes(clutch_size, fill = species)) +
  geom_histogram(binwidth = 1, colour = "white", alpha = 0.9) +
  facet_wrap(~species, ncol = 1, scales = "free_y") +
  scale_fill_manual(values = species_pal, guide = "none") +
  labs(title = "Clutch-size distribution", subtitle = "Most clutches are 6-12 eggs",
       x = "Clutch size (eggs)", y = "Number of nests") +
  theme_ndc()
p_clutch

*Clutches centre on 8–9 eggs with a long single-digit tail — nothing unusual.*

### 3c · When do they lay?
The key plot for our question. Each line counts the nests started in ~5-day windows through the spring.

In [ ]:
p_laydate <- ggplot(nest, aes(ld_doy, colour = species)) +
  geom_freqpoly(binwidth = 5, linewidth = 1) +
  scale_colour_manual(values = species_pal, name = NULL) +
  labs(title = "Timing of egg-laying",
       subtitle = "Number of nests started, in 5-day windows",
       x = "Lay date (day of year)", y = "Number of nests") +
  theme_ndc()
p_laydate

*Laying peaks in mid-late April (day ~100–120). That spring window is exactly where we will look for a greenness signal.*

### 3d · Where are the boxes?
The boxes sit at fixed spots. Their locations matter because in Section 8 we read the greenness right at each box from the NDVI map. Click a marker to see its details.

In [ ]:
nest_sf <- st_as_sf(nest, coords = c("lon_dd", "lat_dd"), crs = 4326, remove = FALSE)

m <- leaflet(nest_sf, width = "100%", height = "520px") %>%
  addProviderTiles("CartoDB.Positron") %>%
  addCircleMarkers(lng = ~lon_dd, lat = ~lat_dd, radius = 4, stroke = FALSE,
                   fillOpacity = 0.8,
                   color = ~unname(species_pal[as.character(species)]),
                   popup = ~paste0("<b>Box:</b> ", nestbox_id,
                                   "<br><b>Species:</b> ", species,
                                   "<br><b>Year:</b> ", year,
                                   "<br><b>Clutch:</b> ", clutch_size,
                                   "<br><b>Lay date:</b> ", lay_date)) %>%
  addLegend("bottomright", colors = unname(species_pal),
            labels = names(species_pal), title = "Species", opacity = 1)

# Workaround for small map height
tmp <- tempfile(fileext = ".html")
htmlwidgets::saveWidget(m, tmp, selfcontained = TRUE)
IRdisplay::display_html(
  paste0('<iframe src="data:text/html;base64,', base64enc::base64encode(tmp), '" width="', m$width, '" height="', m$height,'"></iframe>')
)

<div style="background:#eef3fb;border-left:5px solid #2f6db5;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#2f6db5">Your feedback</b><br>
Before opening the NDC: what environmental context would you most want it to provide for this area?
</div>

## 4 · Visit the Nature Data Cube — *go get the greenness*
Now we leave the birds for a moment and fetch the environment. Open the NDC and **mimic** a download of NDVI for our area:

&nbsp;&nbsp;🔗 **[lter-life-experience.org/naturedatacube](https://lter-life-experience.org/naturedatacube/)**

1. Select project: **Nestboxes**
2. Zoom to Arnhem; select the polygon north of the A12
3. Choose **Biosphere**, then the **NDVI** (greenness) dataset
4. Select **Geodata**
5. Set the period: **2019-01 → 2025-12**
6. **Add to overview**

<div style="background:#fbf3e4;border-left:5px solid #cf9b3f;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#cf9b3f">Note</b><br>
For this exercise you do <b>not</b> actually need to download anything — the data is already saved next to this notebook. Just go through the motions so you can judge the workflow.
</div>

## 5 · What the download contains — *read the receipt*
Every NDC download comes with a download summary listing exactly what you got. Let's read it.

In [ ]:
download_summary <- read.csv(f_summary, stringsAsFactors = FALSE)
download_summary[, c("dataset", "view", "file_type", "status", "note")]

For this area we received (all three `ok`):
- **NDVI Statistics** — a table of monthly mean greenness → *Section 6*
- **NDVI Geodata** — monthly NDVI raster layers → *Section 7*
- the **reference polygon** — the area outline

## 6 · Statistical NDVI outputs — *greenness as a number*
<div style="background:#fbf3e4;border-left:5px solid #cf9b3f;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#cf9b3f">Note</b><br>
<b>Story so far:</b> we have the birds (Section 3) and we have fetched greenness (Section 4). To connect them, we now need to <i>see</i> greenness. The NDC gives it to us two ways — as numbers (this section) and as a map (Section 7). We start with the numbers.
</div>
This first output is a table: one average greenness value per month for the whole area.

In [ ]:
ndvi <- read.csv(f_ndvi, stringsAsFactors = FALSE)
ndvi$date    <- ymd(paste0(ndvi$month, "-01"))
ndvi$yr      <- year(ndvi$date)
ndvi$mon     <- month(ndvi$date)
ndvi$mon_lab <- month(ndvi$date, label = TRUE)
ndvi <- ndvi[order(ndvi$date), ]

### 6a · Monthly NDVI through time
The raw series, with a shaded band for how variable greenness is within the area each month.

In [ ]:
p_ts <- ggplot(ndvi, aes(date, ndvi_mean)) +
  geom_ribbon(aes(ymin = ndvi_mean - ndvi_std, ymax = ndvi_mean + ndvi_std),
              fill = "#009E73", alpha = 0.15) +
  geom_line(colour = "#009E73", linewidth = 0.8) +
  geom_point(colour = "#009E73", size = 1.4) +
  scale_y_continuous(limits = c(0, 1)) +
  labs(title = "Monthly NDVI for the nest-box area",
       subtitle = "Shaded band = +/- 1 SD within the area",
       x = NULL, y = "NDVI (greenness)") +
  theme_ndc()
p_ts

*A clean seasonal cycle: low in winter (\~0.5), peaking in summer (\~0.77). Small gaps are months the satellite was clouded out.*

### 6b · The typical year
Stacking all years shows the *typical* pattern. The steep March→May rise is **green-up** — the breeding-relevant window.

In [ ]:
ndvi_clim <- aggregate(ndvi_mean ~ mon + mon_lab, data = ndvi, FUN = mean)
names(ndvi_clim)[names(ndvi_clim) == "ndvi_mean"] <- "ndvi"
ndvi_clim <- ndvi_clim[order(ndvi_clim$mon), ]
p_season <- ggplot(ndvi_clim, aes(mon_lab, ndvi, group = 1)) +
  geom_line(colour = "#009E73", linewidth = 0.9) +
  geom_point(colour = "#009E73", size = 2) +
  scale_y_continuous(limits = c(0, 1)) +
  labs(title = "Typical greenness through the year",
       subtitle = "Averaged over 2019-2025", x = NULL, y = "Mean NDVI") +
  theme_ndc()
p_season

### 6c · One "spring greenness" number per year
<div style="background:#fbf3e4;border-left:5px solid #cf9b3f;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#cf9b3f">Note</b><br>
<b>Why this step:</b> in Section 8 we want to ask <i>"in greener springs, do birds lay earlier?"</i> To do that we need <b>one number per year</b> that captures how green that spring was. So for each year we average the spring months (March, April, May) into a single value — the year's <b>spring NDVI</b>. The bars below are simply those seven yearly averages (the y-axis is zoomed in because the differences between years are small, but real).
</div>

In [ ]:
spring_rows <- ndvi[ndvi$mon %in% c(3, 4, 5), ]
spring_ndvi <- aggregate(ndvi_mean ~ yr, data = spring_rows, FUN = mean)
names(spring_ndvi)[names(spring_ndvi) == "ndvi_mean"] <- "spring_ndvi"
p_spring <- ggplot(spring_ndvi, aes(factor(yr), spring_ndvi)) +
  geom_col(fill = "#009E73", alpha = 0.85) +
  coord_cartesian(ylim = range(spring_ndvi$spring_ndvi) + c(-0.02, 0.02)) +
  labs(title = "Early-spring greenness by year",
       subtitle = "Average March-May NDVI (whole area) — one value per year",
       x = NULL, y = "Spring NDVI") +
  theme_ndc()
p_spring

## 7 · Geospatial NDVI — *greenness as a map*
The second NDC output is a raster: greenness for every 10 m pixel, every month, already in NDVI units (0–1). The table told us greenness over **time**; the map shows it over **space**.

First we load the raster and build one average map per season.

In [ ]:
ndvi_r <- rast(f_raster)

# Which calendar month does each layer belong to? (NDVI_YYYYMM -> the "MM" part)
lyr_mon <- as.integer(substr(names(ndvi_r), 10, 11))

# Clip to the reference polygon so the map shows only the study area
area_poly <- st_read(f_polygon, quiet = TRUE)
poly_utm  <- st_transform(area_poly, crs(ndvi_r))
ndvi_r    <- mask(crop(ndvi_r, vect(poly_utm)), vect(poly_utm))

# One average map per SEASON
season_months <- list(
  "Winter (Dec-Feb)" = c(12, 1, 2),
  "Spring (Mar-May)" = c(3, 4, 5),
  "Summer (Jun-Aug)" = c(6, 7, 8),
  "Autumn (Sep-Nov)" = c(9, 10, 11)
)
season_maps <- lapply(season_months, function(mm) {
  app(ndvi_r[[which(lyr_mon %in% mm)]], mean, na.rm = TRUE)
})

# One shared green colour scale across all seasons, for an honest comparison
rng <- range(unlist(lapply(season_maps, function(r) values(r))), na.rm = TRUE)
pal <- colorNumeric(c("#ffffe5","#d9f0a3","#78c679","#238443","#004529"),
                    domain = rng, na.color = "transparent")

# Build the layered map (one season per layer) + the nest boxes, then show it.
# Use the control (top-right) to step through the seasons and toggle the boxes.
m <- leaflet(width = "100%", height = 520) %>% addProviderTiles("CartoDB.Positron")
for (nm in names(season_maps)) {
  m <- addRasterImage(m, season_maps[[nm]], colors = pal,
                      opacity = 0.85, project = TRUE, group = nm)
}
m <- m %>%
  addCircleMarkers(data = nest_sf, lng = ~lon_dd, lat = ~lat_dd,
                   radius = 3, stroke = TRUE, weight = 1, color = "black",
                   fillColor = ~unname(species_pal[as.character(species)]),
                   fillOpacity = 0.9, group = "Nest boxes",
                   popup = ~paste0("<b>Box:</b> ", nestbox_id,
                                   "<br><b>Species:</b> ", species)) %>%
  addLegend("bottomright", pal = pal, values = rng,
            title = "Mean NDVI", opacity = 1) %>%
  addLayersControl(
    baseGroups = names(season_months),
    overlayGroups = "Nest boxes",
    options = layersControlOptions(collapsed = FALSE))

# Workaround for small map height
tmp <- tempfile(fileext = ".html")
htmlwidgets::saveWidget(m, tmp, selfcontained = TRUE)
IRdisplay::display_html(
  paste0('<iframe src="data:text/html;base64,', base64enc::base64encode(tmp), '" width="', m$width, '" height="', m$height,'"></iframe>')
)

*Watch the whole area brighten from winter to summer — and notice that some patches stay greener than others all year (denser / evergreen canopy).*

## 8 · Do greener conditions relate to breeding? — *the payoff*
<div style="background:#fbf3e4;border-left:5px solid #cf9b3f;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#cf9b3f">Note</b><br>
We now hold both halves: the <b>birds</b> (Section 3) and greenness as <b>numbers</b> (Section 6) and a <b>map</b> (Section 7). Time to put them together. We look through two lenses, because they answer different things:<br>&nbsp;&nbsp;• <b>between years</b> — is a green spring an early spring?<br>&nbsp;&nbsp;• <b>between species</b> — do the three birds differ, and is that about greenness?
</div>

### 8a · Between years — a weather / phenology signal
Line up each year's spring greenness (from 6c) against how early the birds laid that year.

<div style="background:#fbf3e4;border-left:5px solid #cf9b3f;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#cf9b3f">Note</b><br>
With only <b>seven years</b> this is <b>illustrative</b>, not proof — we show it to demonstrate the kind of analysis the NDC enables.
</div>

In [ ]:
agg_mean <- aggregate(ld_doy ~ year, data = nest, FUN = mean)
agg_n    <- aggregate(ld_doy ~ year, data = nest, FUN = length)
annual   <- data.frame(yr           = agg_mean$year,
                       mean_lay_doy = agg_mean$ld_doy,
                       n_nests      = agg_n$ld_doy)
annual   <- merge(annual, spring_ndvi, by = "yr")

ct_t  <- cor.test(annual$spring_ndvi, annual$mean_lay_doy)
note_t <- sprintf("r = %.2f, p = %.2f (n = %d years) — ILLUSTRATIVE ONLY",
                  ct_t$estimate, ct_t$p.value, nrow(annual))

p_t <- ggplot(annual, aes(spring_ndvi, mean_lay_doy)) +
  geom_smooth(method = "lm", formula = y ~ x, se = TRUE, colour = "grey40",
              fill = "grey80", linetype = "dashed") +
  geom_point(aes(size = n_nests), colour = "#0072B2", alpha = .85) +
  geom_text(aes(label = yr), vjust = -1.1, size = 3, colour = "grey30") +
  scale_size_continuous(name = "Nests in that year", range = c(4, 10)) +
  labs(title = "Between years: greener springs, earlier laying",
       subtitle = note_t, x = "Area-mean spring NDVI (Mar-May)",
       y = "Mean lay date (DOY)") +
  theme_ndc()
p_t

*The slope leans the expected way — greener springs, earlier laying. Direction, not proof.*

### 8b · Between species — is timing about greenness, or about the bird?
Using the **map**, we read the spring greenness at each box, then ask: do the three species nest in *different* greenness, and does that explain their very different laying times?

In [ ]:
box_loc <- unique(nest[, c("nestbox_id", "lon_dd", "lat_dd")])
box_sf  <- st_as_sf(box_loc, coords = c("lon_dd", "lat_dd"), crs = 4326)
box_sf  <- st_transform(box_sf, crs(ndvi_r))
spring_stack <- ndvi_r[[which(lyr_mon %in% c(3, 4, 5))]]
ex <- terra::extract(spring_stack, vect(box_sf))            # ID + spring layers
box_loc$box_spring_ndvi <- rowMeans(ex[, -1], na.rm = TRUE) # mean spring NDVI/box

# attach each box's greenness to its nests, then summarise per species (base R)
nest_ndvi <- merge(nest, box_loc[, c("nestbox_id", "box_spring_ndvi")],
                   by = "nestbox_id", all.x = TRUE)
species_summary <- do.call(rbind, lapply(
  split(nest_ndvi, nest_ndvi$species),
  function(d) data.frame(
    species          = d$species[1],
    mean_spring_ndvi = mean(d$box_spring_ndvi, na.rm = TRUE),
    mean_lay_doy     = mean(d$ld_doy),
    mean_clutch      = mean(d$clutch_size),
    n_nests          = nrow(d))))
rownames(species_summary) <- NULL
species_summary

One point per species: x = greenness it nests in, y = when it lays, point **size = clutch**. The x-axis is on a real NDVI scale, so you can see how *similar* the three are in greenness despite very different timing.

In [ ]:
p_sp <- ggplot(species_summary, aes(mean_spring_ndvi, mean_lay_doy)) +
  geom_point(aes(size = mean_clutch, colour = species), alpha = 0.9) +
  geom_text(aes(label = species), vjust = -1.8, size = 3.5, colour = "grey25") +
  scale_colour_manual(values = species_pal, guide = "none") +
  scale_size_continuous(name = "Mean clutch (eggs)", range = c(6, 14)) +
  coord_cartesian(xlim = c(0.45, 0.75)) +
  labs(title = "Between species: same greenness, very different timing",
       subtitle = "Each point a species — similar greenness, but ~18 days apart in laying",
       x = "Mean spring NDVI where the species nests",
       y = "Mean lay date (DOY)") +
  theme_ndc()
p_sp

### 8c · What the two lenses tell us
<div style="background:#fbf3e4;border-left:5px solid #cf9b3f;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#cf9b3f">Note</b><br>
<b>Between years</b>, greenness tracks timing: a greener spring is an earlier spring for everyone — a weather / phenology response.<br><br><b>Between species</b>, the picture flips: blue tits, great tits and pied flycatchers all nest in almost the <b>same</b> greenness, yet pied flycatchers lay ~18 days later and ~3 fewer eggs than blue tits. Those differences are about the <b>bird</b> (its life history), not the greenness of where it nests.<br><br><b>Together:</b> greenness helps explain change over <i>time</i>, not the differences <i>between</i> species.
</div>

## 9 · Your overall feedback
<div style="background:#eef3fb;border-left:5px solid #2f6db5;padding:12px 16px;border-radius:6px;margin:10px 0;">
<b style="color:#2f6db5">Your feedback</b><br>
Please jot down a few thoughts:<br><br><b>1. Workflow</b> — Did exploring the NDC feel intuitive? Where did it not?<br><b>2. Clarity</b> — Did the statistics and the map communicate the data clearly?<br><b>3. Missing</b> — What information or output did you expect but not find?<br><b>4. Trust</b> — Would you rely on these outputs in your research? Why / why not?<br><b>5. One change</b> — If you could change one thing about the NDC, what would it be?
</div>
Thank you for helping us improve the Nature Data Cube.